# 061 — Round 4: grouped k-fold evaluation

Reads whatever `060_kfold_training.ipynb` has produced under `models/kfold/fold_*/`
and reports:

- **§2 — per-fold held-out pixel metrics** (`mae`/`psnr`/`ssim` on that fold's
  held-out real artworks, scored by that fold's own model) → **mean ± std** across
  folds. This is the variance estimate `fixing.md` #4 is about.
- **§3 — detection on `GT01`-`GT03`**: `structural delta` and `structural z` AUROC +
  stroke coherence, **per fold-model** and for the **3-fold ensemble** (mean of the
  fold `μ`s). The GT paintings are external to every fold, so each fold-model is a
  genuine held-out predictor for them.
- **§4 — headline** numbers for the write-up.

Runs with **≥ 1 fold**; with `< K` folds the mean±std is over what exists, with a
warning. Re-run as more folds finish.

Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports — the `C4`/`C5` detection toolkit plus the k-fold split.

In [ ]:
import gc

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image

from scripts.calibration import laplace_sigma_from_scale, learned_zscore, structural_zscore
from scripts.config import settings
from scripts.dataset import build_dataset, load_image_pairs, pad_to_multiple
from scripts.delta_analysis import analyze_delta
from scripts.detection import evaluate_detection
from scripts.kfold import fold_artwork_groups, grouped_kfold_splits
from scripts.stroke_stats import stroke_coherence
from scripts.trainer_nll import load_model_nll
from scripts.visualization import plot_gt_signal_gallery

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")

## 1. Which folds are available

In [ ]:
ARCH = "attention_unet_nll"
K = settings.KFOLD_K
KFOLD_DIR = settings.MODELS_DIR / "kfold"

pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
held_out = fold_artwork_groups(pairs, k=K, seed=settings.KFOLD_SEED)
splits = grouped_kfold_splits(pairs, k=K, seed=settings.KFOLD_SEED)

available = [
    f for f in range(K)
    if (KFOLD_DIR / f"fold_{f}" / ARCH / "best_model.keras").exists()
]
print(f"folds trained: {available}  ({len(available)}/{K})")
if not available:
    raise RuntimeError("No fold checkpoints under models/kfold/ — run 060 first.")
if len(available) < K:
    print(f"WARNING: {K - len(available)} fold(s) missing — statistics are over "
          f"{len(available)} fold(s) only. Re-run when more finish.")

## 2. Per-fold held-out pixel metrics

Each fold's model evaluated on its own held-out real artworks (full-resolution,
`batch_size=1`). The spread across folds is the point of the round.

In [ ]:
pixel_rows = {}
for fold in available:
    model = load_model_nll(
        ARCH, model_dir=KFOLD_DIR / f"fold_{fold}",
        loss_name="laplace_nll", beta=settings.NLL_BETA,
    )
    _, val_pairs = splits[fold]
    val_ds = build_dataset(val_pairs, batch_size=1, augment=False, shuffle=False)
    scores = dict(zip(model.metrics_names, model.evaluate(val_ds, verbose=0)))
    pixel_rows[fold] = scores
    print(f"fold {fold}: " + "  ".join(f"{k}={v:.4f}" for k, v in scores.items()))
    del model, val_ds
    gc.collect()
    tf.keras.backend.clear_session()

keys = [k for k in ("loss", "mae", "psnr", "ssim") if k in next(iter(pixel_rows.values()))]
print(f"\n{'metric':<10}" + "".join(f"fold {f}".rjust(12) for f in available) + f"{'mean':>12}{'std':>10}")
print("-" * (10 + 12 * len(available) + 22))
for k in keys:
    vals = np.array([pixel_rows[f][k] for f in available])
    print(f"{k:<10}" + "".join(f"{pixel_rows[f][k]:.4f}".rjust(12) for f in available)
          + f"{vals.mean():>12.4f}{vals.std():>10.4f}")

## 3. Detection on the GT paintings

`GT01`/`GT02`/`GT03` are outside every fold, so each fold-model is a held-out
predictor for them. Signals: `structural delta` (`1 - local SSIM structure` of the
fold `μ`) and `structural z` (that, divided by the fold's learned `σ`). Scored per
fold-model and for the **ensemble** = the pixel-wise mean of the available fold `μ`s
(and, for `structural z`, the mean of their `σ`s).

In [ ]:
TEST_RGB = project_root / "data" / "test" / "rgb"
TEST_IR = project_root / "data" / "test" / "ir"
ANN = project_root / "data" / "test" / "annotations"
GT_STEMS = sorted(p.name.removesuffix("_Map.png") for p in ANN.glob("*_Map.png"))


def _find(stem, folder):
    for ext in (".jpg", ".png", ".jpeg", ".tif", ".tiff"):
        if (folder / f"{stem}{ext}").exists():
            return folder / f"{stem}{ext}"
    return None


def load_triplet(stem):
    rgb = np.array(Image.open(_find(stem, TEST_RGB)).convert("RGB"), np.float32) / 255.0
    ir = np.array(Image.open(_find(stem, TEST_IR)).convert("L"), np.float32) / 255.0
    mask = np.array(Image.open(ANN / f"{stem}_Map.png").convert("L")) > 127
    return rgb, ir, mask


def predict(model, rgb, hw):
    padded, _ = pad_to_multiple(tf.constant(rgb), multiple=settings.PATCH_MULTIPLE)
    h, w = hw
    p = model.predict(padded[tf.newaxis, ...], verbose=0)[0, :h, :w, :]
    return p[..., 0], laplace_sigma_from_scale(np.exp(p[..., 1]))


# gt_data[stem] = (rgb, ir, mask); preds[stem][fold] = (mu, sigma)
gt_data = {s: load_triplet(s) for s in GT_STEMS}
preds = {s: {} for s in GT_STEMS}

for fold in available:
    model = load_model_nll(
        ARCH, model_dir=KFOLD_DIR / f"fold_{fold}",
        loss_name="laplace_nll", beta=settings.NLL_BETA,
    )
    for s in GT_STEMS:
        rgb, ir, _ = gt_data[s]
        preds[s][fold] = predict(model, rgb, ir.shape)
    del model
    gc.collect()
    tf.keras.backend.clear_session()

print(f"predicted {GT_STEMS} with folds {available}")

In [ ]:
def signals_from(ir, mu, sigma):
    d = analyze_delta(ir, mu)
    return {
        "structural delta": d.structural_delta,
        "structural z": structural_zscore(d.structural_delta, sigma),
    }


def score(name, sig, mask):
    r = evaluate_detection(sig, mask)
    return r.auroc, stroke_coherence(sig).coherence


# rows[(scope, signal)] = {stem: (auroc, coherence)}
rows = {}
for s in GT_STEMS:
    rgb, ir, mask = gt_data[s]
    # per fold
    for fold in available:
        mu, sigma = preds[s][fold]
        for name, sig in signals_from(ir, mu, sigma).items():
            rows.setdefault((f"fold {fold}", name), {})[s] = score(name, sig, mask)
    # ensemble
    mu_e = np.mean([preds[s][f][0] for f in available], axis=0)
    sig_e = np.mean([preds[s][f][1] for f in available], axis=0)
    for name, sig in signals_from(ir, mu_e, sig_e).items():
        rows.setdefault(("ensemble", name), {})[s] = score(name, sig, mask)

for signal in ("structural delta", "structural z"):
    print(f"\n=== {signal} ===")
    print(f"{'scope':<12}" + "".join(f"{s} AUROC".rjust(14) for s in GT_STEMS)
          + f"{'mean AUROC':>12}{'mean coh':>10}")
    print("-" * (12 + 14 * len(GT_STEMS) + 22))
    for scope in [f"fold {f}" for f in available] + ["ensemble"]:
        d = rows[(scope, signal)]
        au = np.array([d[s][0] for s in GT_STEMS])
        co = np.array([d[s][1] for s in GT_STEMS])
        print(f"{scope:<12}" + "".join(f"{d[s][0]:.3f}".rjust(14) for s in GT_STEMS)
              + f"{au.mean():>12.3f}{co.mean():>10.3f}")

## 4. Headline

The numbers for `evaluation.md` §4b: mean ± std of the mean-over-GT AUROC **across
fold-models**, and the ensemble's value.

In [ ]:
print(f"model: {ARCH}   folds: {available}   (GT mean over {GT_STEMS})\n")
for signal in ("structural delta", "structural z"):
    per_fold = np.array([
        np.mean([rows[(f"fold {f}", signal)][s][0] for s in GT_STEMS])
        for f in available
    ])
    ens = np.mean([rows[("ensemble", signal)][s][0] for s in GT_STEMS])
    print(f"{signal:<18} AUROC per fold {np.round(per_fold, 3).tolist()}  "
          f"-> {per_fold.mean():.3f} ± {per_fold.std():.3f}   ensemble {ens:.3f}")

for k in [x for x in ("psnr", "ssim", "mae") if pixel_rows and x in next(iter(pixel_rows.values()))]:
    v = np.array([pixel_rows[f][k] for f in available])
    print(f"{k:<18} held-out per fold {np.round(v, 4).tolist()}  -> {v.mean():.4f} ± {v.std():.4f}")

## 5. Visual — the ensemble signal on each GT painting

In [ ]:
for s in GT_STEMS:
    rgb, ir, mask = gt_data[s]
    mu_e = np.mean([preds[s][f][0] for f in available], axis=0)
    sig_e = np.mean([preds[s][f][1] for f in available], axis=0)
    panels = signals_from(ir, mu_e, sig_e)
    fig = plot_gt_signal_gallery(rgb, ir, mask, panels, title=f"061 — {s} — {len(available)}-fold ensemble")
    plt.show()
    plt.close(fig)

## 6. Conclusion

_Fill in after all folds are trained._

- **Pixel-metric spread across folds** (§2): std on psnr / ssim / mae = … Is the
  single-split point estimate from `030`/`033` within ±1 std of the k-fold mean? …
- **Detection AUROC spread** (§3/§4): `structural z` = … ± … ; `structural delta` =
  … ± … . Does the fold-to-fold variance change which signal looks better? …
- **Ensemble** vs. the best single fold — better, same, worse? …
- **Finding #4 verdict**: does cross-validation confirm the single-split metrics were
  trustworthy (small std) or not (large std)? …

**For the write-up (`evaluation.md` §4b, `fixing.md` §0):** report the k-fold
mean ± std as the headline number, note the ensemble result, and state whether #4 is
resolved as "single split was adequate" or "single split was optimistic/noisy".